# Nuclei Lab · Google Colab walkthrough

Choose **Runtime → Run all**. The setup cell downloads the repository and installs dependencies automatically. A standard **CPU runtime** is sufficient. No Google Drive access or full dataset download is required.

This notebook segments nuclei in fluorescence images with classical computer vision. The examples are BBBC039 validation images; evaluation results come from a separate test split. Streamlit is an optional local app and is not needed here.

In [ ]:
# Run this cell first. It also works in a fresh Google Colab CPU runtime.
from pathlib import Path
import subprocess
import sys
import tempfile

REPOSITORY_URL = "https://github.com/virzinho/cell-counter-qc.git"
local_candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in local_candidates
             if (p / "cell_counter/segmentation.py").is_file()
             and (p / "examples/manifest.json").is_file()), None)
if ROOT is None:
    ROOT = Path(tempfile.mkdtemp(prefix="nuclei-lab-")) / "cell-counter-qc"
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(ROOT)], check=True)
# Install only the scientific core: Streamlit is optional for local use.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))
print("Ready:", ROOT)
print("Continue with the next cells, or choose Runtime > Run all.")


## 1. Load a real microscopy image
Choose example 1, 2 or 3. The original 16-bit TIFF is preserved before normalization.

In [ ]:
import json
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from scipy.ndimage import gaussian_filter
from cell_counter import Config, segment, measure, normalize, overlay, quality
from cell_counter.dataset import read_image

EXAMPLE = 1 # @param {type:"slider", min:1, max:3, step:1}
manifest = json.loads((ROOT / "examples/manifest.json").read_text())
item = manifest[int(EXAMPLE) - 1]
image = read_image(ROOT / "examples" / item["file"])
reference_count = item["reference_count"]
print("Source:", item["source_filename"])
print("Image shape:", image.shape, "| dtype:", image.dtype)
print("Reference annotation:", reference_count, "nuclei")
plt.figure(figsize=(9, 6))
plt.imshow(normalize(image), cmap="gray"); plt.axis("off"); plt.title("BBBC039 validation image"); plt.show()


### Optional: use your own image
Leave the option off for uninterrupted Run all. For uploads, use a single 2D image with bright fluorescent nuclei on a dark background.

In [ ]:
# Leave this False to run the whole notebook without any interactive prompts.
USE_OWN_IMAGE = False # @param {type:"boolean"}
if USE_OWN_IMAGE:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("This upload option requires Google Colab. For local use, set image = read_image(your_path).")
    uploaded = files.upload()
    if uploaded:
        filename, payload = next(iter(uploaded.items()))
        if len(payload) > 10 * 1024**2:
            raise ValueError("Please use an image smaller than 10 MB")
        with Image.open(io.BytesIO(payload)) as opened:
            if opened.width * opened.height > 4_000_000:
                raise ValueError("Please use an image with at most four million pixels")
            if getattr(opened, "n_frames", 1) != 1:
                raise ValueError("Export one 2D frame from the image stack first")
            image = np.array(opened)
        normalize(image)
        reference_count = None
        print("Using uploaded image:", filename)


## 2. Segment and count nuclei
Parameters were selected on validation images. Watershed separates some touching regions using distance-transform markers. The count is an estimate, not a validated diagnosis.

In [ ]:
selected = json.loads((ROOT / "reports/selected_config.json").read_text())
config = Config(**selected["watershed"])
labels = segment(image, "watershed", config)
print("Detected nuclei:", int(labels.max()))
if reference_count is not None:
    print("Reference count:", reference_count, "| count difference:", int(labels.max()) - reference_count)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), layout="constrained")
axes[0].imshow(normalize(image), cmap="gray"); axes[0].set_title("Input image")
axes[1].imshow(overlay(image, labels)); axes[1].set_title(f"Watershed: {labels.max()} nuclei")
for ax in axes: ax.axis("off")
plt.show()


## 3. Inspect and export morphology measurements
Each row represents a predicted nucleus. Areas are in px² and lengths in pixels.

In [ ]:
table = measure(labels)
display(table.head(10))
print("Measurements use pixels, not calibrated physical units.")
OUTPUT = ROOT / "outputs/colab"
OUTPUT.mkdir(parents=True, exist_ok=True)
table.to_csv(OUTPUT / "measurements.csv", index=False)
Image.fromarray(labels.astype(np.uint16)).save(OUTPUT / "labels.tif")
Image.fromarray((overlay(image, labels) * 255).astype(np.uint8)).save(OUTPUT / "overlay.png")
print("Exported measurements, label image and overlay:", OUTPUT)


## 4. Test image-quality changes
Compare the same image after digital blur or added noise. This example does not calculate ground-truth F1; the full benchmark does.

In [ ]:
x = normalize(image)
noisy = np.clip(x + np.random.default_rng(42).normal(0, 0.08, x.shape), 0, 1)
cases = {"Original": x, "Blur sigma=4": gaussian_filter(x, 4), "Noise sigma=.08": noisy}
rows = []
fig, axes = plt.subplots(1, 3, figsize=(16, 5), layout="constrained")
for ax, (name, altered) in zip(axes, cases.items()):
    prediction = segment(altered, "watershed", config)
    rows.append({"condition": name, "predicted_count": int(prediction.max()),
                 "count_change": int(prediction.max()) - int(labels.max()), **quality(altered)})
    ax.imshow(overlay(altered, prediction)); ax.set_title(f"{name}: {prediction.max()} nuclei"); ax.axis("off")
plt.show()
display(pd.DataFrame(rows))
print("Quality indicators are descriptive. Noise can increase apparent sharpness.")


## 5. Read the recorded held-out benchmark
50 official test images, 6 image conditions, 2 methods. Instance F1 matches nucleus regions one-to-one at IoU ≥ 0.5. Dice only measures foreground overlap.

In [ ]:
results = pd.read_csv(ROOT / "reports/summary.csv")
display(results[["method", "condition", "n_images", "absolute_count_error", "instance_f1", "foreground_dice"]])
display(Image.open(ROOT / "reports/robustness.png"))
print("These are recorded held-out results, not metrics calculated from the demonstration image.")


## 6. Download your output files
Set DOWNLOAD_RESULTS to True and rerun the next cell to download the ZIP in Colab.

In [ ]:
archive = OUTPUT / "nuclei_lab_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for filename in ["measurements.csv", "labels.tif", "overlay.png"]:
        bundle.write(OUTPUT / filename, arcname=filename)
print("Results archive:", archive)
DOWNLOAD_RESULTS = False # @param {type:"boolean"}
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive))


## Optional: reproduce the entire experiment
This takes longer and downloads the source dataset. It is disabled by default.

In [ ]:
# Optional: downloads about 81 MB and reruns validation selection + test evaluation.
RUN_FULL_BENCHMARK = False # @param {type:"boolean"}
if RUN_FULL_BENCHMARK:
    subprocess.run([sys.executable, str(ROOT / "scripts/download_data.py")], cwd=ROOT, check=True)
    subprocess.run([sys.executable, str(ROOT / "scripts/benchmark.py")], cwd=ROOT, check=True)
else:
    print("Full benchmark skipped. The bundled examples and recorded results are sufficient for this walkthrough.")


## Interpretation and limits

The dataset covers one U2OS screen, not every cell line or microscope. Dim and touching nuclei can be missed or split incorrectly. Digital blur is not a full optical model. Read `docs/METHODS.md` and `docs/PORTFOLIO.md` in the repository for the protocol, limitations and interview notes.